# GPU04 — خ۶ (F06): فرایند گاوسی (GP) روی L1 با GPyTorch

> بند 7.15 `doc/WBS-phase7-modeling.md` · اسپرینت C، ردیف «خ۶ کرنل: GP».

جدول 7.15.4 می‌گوید GP روی L1 (۷٬۵۷۹ نقطه) «در مرز» است و روی CPU ~۱۰ دقیقه
می‌گیرد. روی GPU همان کار ثانیه‌ای است — پس دلیل محدودکردن به L3 (۲۵۶ نقطه) از بین
می‌رود و GP روی **همان سطحی** اجرا می‌شود که همه‌ی خانواده‌های دیگر با آن سنجیده
شدند (بند 7.1.2). این خودش یکی از دلایل فرستادن این خانواده به GPU است.

**دو خروجی اجباری بند 7.15.6 که این نوت‌بوک تولید می‌کند:**

1. **جدول مقایسه‌ی ۷ ترکیب کرنل** (K1…K7 جدول 7.15.3) با درست‌نمایی حاشیه‌ای لگاریتمی.
2. **طول‌مقیاس ARD هر فیچر** — خروجی تفسیری: طول‌مقیاس کوچک ⇒ فیچر مهم. خودش یک
   روش انتخاب فیچر است (بند 7.5.4).

⭐ K7 (`gp_heteroscedastic`) واریانس نویز را تابع $\log Res$ می‌کند — **پاسخ مستقیم
به F06** (ناهم‌واریانسی تأییدشده، نسبت std چارک کوچک به بزرگ ۳.۰۳).

**بودجه‌ی هدف: ~۹۵ دقیقه.**

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [1]:
!pip install -q gpytorch optuna mlflow tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 679.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [2]:
MODE = "kaggle"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/working/t.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
# with zipfile.ZipFile(bundle) as z:
#     z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))

import os
import shutil

src_dir = "/kaggle/input/datasets/mvajhi/bundle"
dest_dir = "/kaggle/working/phase7"

os.makedirs(dest_dir, exist_ok=True)

# Copy all contents from Kaggle input to working/my_dir
for item in os.listdir(src_dir):
    src_path = os.path.join(src_dir, item)
    dest_path = os.path.join(dest_dir, item)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dest_path)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

محتوای بسته: ['AGENTS.md', 'BUNDLE_INFO.json', 'data', 'doc', 'reports', 'requirements-gpu.txt', 'src']


## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [3]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.gpu_runner import load_l1
data = load_l1()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

✅ دروازه‌ی انصاف A1 پاس شد · cv_folds_hash=bd08d6f7c801… · data_snapshot_hash=68b4cb8517d2…
L1 — 7,579 ردیف، 5 fold (fold0: 3,556→870 · fold1: 4,426→860 · fold2: 5,286→185 · fold3: 5,471→1,025 · fold4: 6,496→940)


## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [4]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [5]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

Fri Aug 21 12:44:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

INFO:2026-08-21 12:44:43,307:jax._src.xla_bridge:822: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-08-21 12:44:43,307 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


{'platform': 'Linux-6.12.90+-x86_64-with-glibc2.35',
 'python': '3.12.13',
 'torch': '2.10.0+cu128',
 'cuda': '12.8',
 'device': 'cuda',
 'gpu_name': 'Tesla T4',
 'gpu_memory_gb': 15.64,
 'jax': '0.7.2',
 'jax_devices': ['cuda:0', 'cuda:1']}

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [6]:
from pathlib import Path
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "kaggle"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

# reports/gpu/ را همین‌جا می‌سازیم — سلول‌های بعدی مستقیم CSV آن‌جا می‌نویسند،
# پیش از آنکه save_family_report/package_outputs بسازدش
Path("reports/gpu").mkdir(parents=True, exist_ok=True)


MLflow → /kaggle/working/phase7/mlruns_gpu


## سلول ۷-الف — R0: آزمایش دود

In [7]:
from src.models.families import f06_kernel as fam
from src.models.gpu_runner import smoke_test

smoke = [smoke_test(fam.FITTERS[m], data, hyperparams={"n_iters": 40})
         for m in ["gp_quantile", "gp_heteroscedastic"]]

R0 gp_quantile                  pinball=0.01641 (B3=0.01375) پوشش=0.186 R²=-0.304 21.0s


/usr/local/lib/python3.12/dist-packages/gpytorch/models/exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


R0 gp_heteroscedastic           pinball=0.01644 (B3=0.01375) پوشش=0.244 R²=-0.173 29.6s


## سلول ۷-ب — ⭐ جدول اجباری ۷.۱۵.۶: مقایسه‌ی هفت ترکیب کرنل

هر ترکیب روی **هر ۵ fold رسمی** برازش می‌شود؛ هم pinball و هم درست‌نمایی حاشیه‌ای
لگاریتمی (LML) ثبت می‌شود. LML معیار انتخاب **درون‌مدلی** GP است و pinball معیار
پروژه — اگر این دو با هم نخوانند، خودش یک یافته است.

⏱ سنگین‌ترین سلول این نوت‌بوک (~۴۰ دقیقه). اگر session ناپایدار است `N_ITERS` را
کم کنید.

In [8]:
import numpy as np, pandas as pd, time
from src.baselines import operational_metrics
from src.models.axes import TUNING_TAU
from src.models.gpu_runner import baseline_b3_per_fold

N_ITERS = 120
b3 = baseline_b3_per_fold(data.folds, TUNING_TAU)

rows = []
for combo in fam.KERNEL_COMBOS:
    het = combo == "K7_heteroscedastic"
    fitter = fam.FITTERS["gp_heteroscedastic" if het else "gp_quantile"]
    t0, pinballs, lmls, covs = time.time(), [], [], []
    try:
        for tr, te in data.folds:
            m = fitter.fit(tr, TUNING_TAU, kernel=("K5_full" if het else combo), n_iters=N_ITERS)
            met = operational_metrics(te, m.predict(te, TUNING_TAU), TUNING_TAU)
            pinballs.append(met["pinball"]); covs.append(met["coverage"]); lmls.append(m.lml)
        rows.append({"ترکیب کرنل": combo, "pinball": round(float(np.mean(pinballs)), 5),
                     "B3": round(b3["mean_pinball"], 5), "پوشش": round(float(np.mean(covs)), 4),
                     "LML (میانگین fold)": round(float(np.mean(lmls)), 2),
                     "ثانیه": round(time.time() - t0, 1)})
        print(f"  {combo:<24s} pinball={np.mean(pinballs):.5f}  LML={np.mean(lmls):8.2f}  "
              f"{time.time()-t0:5.1f}s")
    except Exception as e:
        rows.append({"ترکیب کرنل": combo, "pinball": float("nan"), "خطا": str(e)[:120]})
        print(f"  {combo:<24s} ❌ {type(e).__name__}: {str(e)[:100]}")

kernel_table = pd.DataFrame(rows).sort_values("pinball")
kernel_table.to_csv("reports/gpu/F06_kernel_comparison.csv", index=False)
kernel_table

  K1_rbf                   pinball=0.02062  LML=   -1.06  359.7s
  K2_periodic              pinball=0.02099  LML=   -1.36  306.3s
  K3_rbf_x_periodic        pinball=0.02107  LML=   -1.06  452.2s
  K4_matern                pinball=0.02053  LML=   -1.01  347.1s
  K5_full                  pinball=0.02103  LML=   -1.03  484.7s
  K6_full_plus_linear      pinball=0.02114  LML=   -0.83  629.8s


/usr/local/lib/python3.12/dist-packages/gpytorch/models/exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gpytorch/models/exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gpytorch/models/exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gpytorch/models/exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gpytorch/models/exact_gp.py:299: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


  K7_heteroscedastic       pinball=0.02113  LML=   -1.03  970.5s


,ترکیب کرنل,pinball,B3,پوشش,LML (میانگین fold),ثانیه
3,K4_matern,0.02053,0.0159,0.1300,-1.01,347.1
0,K1_rbf,0.02062,0.0159,0.1023,-1.06,359.7
1,K2_periodic,0.02099,0.0159,0.0968,-1.36,306.3
4,K5_full,0.02103,0.0159,0.0891,-1.03,484.7
2,K3_rbf_x_periodic,0.02107,0.0159,0.1080,-1.06,452.2
6,K7_heteroscedastic,0.02113,0.0159,0.1072,-1.03,970.5
5,K6_full_plus_linear,0.02114,0.0159,0.1826,-0.83,629.8


## سلول ۷-ج — R2: تنظیم با بودجه‌ی زمانی

In [9]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

studies = [
    run_gpu_study(fam.FITTERS["gp_quantile"], SPACES["gp_quantile"].fn, data,
                  family=fam.FAMILY, feature_set=fam.FEATURE_SET,
                  budget_minutes=22, compute=COMPUTE, seed=42),
    run_gpu_study(fam.FITTERS["gp_heteroscedastic"], SPACES["gp_heteroscedastic"].fn, data,
                  family=fam.FAMILY, feature_set=fam.FEATURE_SET,
                  budget_minutes=18, compute=COMPUTE, seed=42),
]

2026/08/21 13:44:46 INFO mlflow.tracking.fluent: Experiment with name 'phase7' does not exist. Creating a new experiment.



R2 — F06/gp_quantile (L1) · τ=0.2 · بودجه=22 دقیقه · دستگاه=cuda · مرجع B3=0.01590


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   0 | pinball=0.02069 | بهترین=0.02069 | 170.3s | گذشته=  2.8/22 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

...

  trial   3 | pinball=0.02060 | بهترین=0.01794 | 483.0s | گذشته= 20.9/22 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   4 | pinball=0.02068 | بهترین=0.01794 | 795.1s | گذشته= 34.1/22 دقیقه
⏱️  بودجه‌ی زمانی (22 دقیقه) تمام شد پس از 5 trial — با بهترین نتیجه‌ی تا این لحظه ادامه می‌دهیم.

✅ gp_quantile: بهترین pinball=0.01794 در برابر B3=0.01590 (باخت) · 5 trial · همگرا(A6)=✅ · پایداری=3/5

R2 — F06/gp_heteroscedastic (L1) · τ=0.2 · بودجه=18 دقیقه · دستگاه=cuda · مرجع B3=0.01590


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   0 | pinball=0.02161 | بهترین=0.02161 | 1288.8s | گذشته= 21.5/18 دقیقه
⏱️  بودجه‌ی زمانی (18 دقیقه) تمام شد پس از 1 trial — با بهترین نتیجه‌ی تا این لحظه ادامه می‌دهیم.

✅ gp_heteroscedastic: بهترین pinball=0.02161 در برابر B3=0.01590 (باخت) · 1 trial · همگرا(A6)=✅ · پایداری=5/5


## سلول ۷-د — قهرمان + ACI + DM

GP قطعی است (بدون نمونه‌گیری تصادفی در برازش)، پس دو seed برای نشان‌دادن پایداری
بهینه‌سازی Adam کافی است — قاعده‌ی سه seed (A7) برای شبکه‌های عصبی نوشته شده.

In [10]:
from src.models.gpu_runner import finalize_champion

best = min(studies, key=lambda s: s.best_pinball)
print(f"قهرمان: {best.model_id} (pinball={best.best_pinball:.5f})\n")
champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234),
                               compute=COMPUTE, run_aci=True)]

قهرمان: gp_quantile (pinball=0.01794)

  seed 42: pinball(ردیفی)=0.01580
  seed 1234: pinball(ردیفی)=0.01578
  ACI: پوشش=0.2000 (شکاف +0.0000) · pinball=0.01519

🏁 gp_quantile: pinball(ردیفی)=0.01578 در برابر B3=0.01335 · DM p=0.0000 ❌ غیرمعنادار · 30 فایل مدل ذخیره شد


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

## سلول ۷-ه — ⭐ خروجی تفسیری اجباری: طول‌مقیاس ARD هر فیچر

طول‌مقیاس کوچک ⇒ خروجی با تغییر آن فیچر سریع عوض می‌شود ⇒ فیچر مهم است. این جدول
مستقیماً با اهمیت فیچر LightGBM (خ۲) قابل‌مقایسه است و یکی از خروجی‌های
گزارشی این خانواده است (بند 7.15.6).

In [11]:
from pathlib import Path

stem = Path(f"models/gpu/F06/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
ard = reloaded.ard_lengthscales()
print(f"کرنل: {reloaded.config['kernel']} · LML={reloaded.lml:.2f} · "
      f"نویز ناهم‌واریانس: {reloaded.noise_model}")
ard.to_csv("reports/gpu/F06_ard_lengthscales.csv", index=False)
ard.head(20)

کرنل: K5_full · LML=-1.08 · نویز ناهم‌واریانس: None


,kernel_param,feature,lengthscale
0,covar_module.kernels.1.base_kernel.raw_lengths...,RestaurantName=ژئوفیزیک,0.268668
1,covar_module.kernels.1.base_kernel.raw_lengths...,RestaurantName=علوم اجتماعی,0.300000
2,covar_module.kernels.0.kernels.0.base_kernel.r...,RestaurantName=بیوشیمی و بیوفیزیک,0.345897
3,covar_module.kernels.0.kernels.0.base_kernel.r...,RestaurantName=علوم اجتماعی,0.381926
4,covar_module.kernels.0.kernels.0.base_kernel.r...,RestaurantName=مطالعات جهان,0.585211
5,covar_module.kernels.1.base_kernel.raw_lengths...,RestaurantName=بیوشیمی و بیوفیزیک,0.623998
6,covar_module.kernels.1.base_kernel.raw_lengths...,is_ramadan,0.693147
7,covar_module.kernels.0.kernels.0.base_kernel.r...,is_ramadan,0.693147
8,covar_module.kernels.0.kernels.0.base_kernel.r...,is_day_before_holiday,0.759269
9,covar_module.kernels.1.base_kernel.raw_lengths...,RestaurantName=روانشناسی و علوم تربیتی,0.798892


## سلول ۷-و — راستی‌آزمایی مدل ذخیره‌شده

⚠️ GP **پارامتری نیست**: بدون داده‌ی آموزش، هایپرپارامترها بی‌فایده‌اند — به همین
دلیل ماتریس آموزش هم داخل همان فایل ذخیره شده. این سلول ثابت می‌کند مدل بازخوانی‌شده
واقعاً پیش‌بینی می‌کند.

In [12]:
import numpy as np
from src.models.axes import TAU_GRID

mean, sd = reloaded.posterior(data.folds[0][1])
print(f"پسین GP — میانگین={mean.mean():.5f} · انحراف معیار میانگین={sd.mean():.5f}")
print("τ | کوانتایل از همان پسین (یک برازش، همه‌ی τها):")
for t in TAU_GRID:
    print(f"{t:.2f} | {reloaded.predict(data.folds[0][1], t).mean():.5f}")
assert np.isfinite(mean).all() and np.isfinite(sd).all()
print("\n✅ مدل ذخیره‌شده قابل استفاده است")

پسین GP — میانگین=0.10662 · انحراف معیار میانگین=0.06340
τ | کوانتایل از همان پسین (یک برازش، همه‌ی τها):
0.02 | 0.00218
0.05 | 0.00946
0.10 | 0.02862
0.15 | 0.04258
0.20 | 0.05415

✅ مدل ذخیره‌شده قابل استفاده است


## سلول ۷-ز — گزارش فارسی کامل

In [13]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    "جدول مقایسه‌ی ۷ ترکیب کرنل (بند 7.15.3/7.15.6): `reports/gpu/F06_kernel_comparison.csv`.",
    "طول‌مقیاس ARD هر فیچر: `reports/gpu/F06_ard_lengthscales.csv` — خروجی تفسیری خانواده.",
    "GP روی L1 اجرا شد نه L3 — روی GPU محدودیت مقیاس‌پذیری بند 7.15.4 عملاً برطرف است.",
    "کوانتایل مستقیم از پسین گاوسی گرفته شد (مسیر Q4)، بدون آفست تجربی باقیمانده.",
    "K7 واریانس نویز را تابع log Res کرد — پاسخ مستقیم به F06؛ نتیجه‌اش در جدول کرنل.",
]
report = render_family_report("F06", "خ۶ — فرایند گاوسی روی L1 (GPyTorch، اجرای GPU)",
                              studies, champions, smoke, DEVICE, notes)
report += ("\n## مقایسه‌ی ترکیب‌های کرنل (بند 7.15.3)\n\n"
           + kernel_table.to_markdown(index=False)
           + "\n\n## ده فیچر با کوچک‌ترین طول‌مقیاس ARD\n\n"
           + ard.head(10).to_markdown(index=False) + "\n")
save_family_report("F06", report, "F06_gp_L1")
print(report)

گزارش نوشته شد: /kaggle/working/phase7/reports/gpu/F06_gp_L1.md
# خ۶ — فرایند گاوسی روی L1 (GPyTorch، اجرای GPU)

> اجرای GPU، بند 7.8 `doc/WBS-phase7-modeling.md`. خانواده F06. سخت‌افزار: Tesla T4 · torch=2.10.0+cu128 · CUDA=12.8

## R0 — آزمایش دود (اجراپذیری + سیم‌چین نشتی)

| مدل | pinball | B3 | پوشش | R² | زمان |
|---|---|---|---|---|---|
| `gp_quantile` | 0.01641 | 0.01375 | 0.186 | -0.304 | 21.0s |
| `gp_heteroscedastic` | 0.01644 | 0.01375 | 0.244 | -0.173 | 29.6s |

## R2 — تنظیم با بودجه‌ی زمانی

| مدل | بهترین pinball | B3 | trial | همگرا (A6) | پایداری (۷.۶.۳) | شکست | ساعت-هسته |
|---|---|---|---|---|---|---|---|
| `gp_quantile` | **0.01794** | 0.01590 | 5 | ✅ | 3/5 | 0 | 0.57h |
| `gp_heteroscedastic` | **0.02161** | 0.01590 | 1 | ✅ | 5/5 | 0 | 0.36h |

## S3 — قهرمان‌ها: سه seed، کالیبراسیون ACI، آزمون Diebold-Mariano

| مدل | pinball(ردیفی) | B3(ردیفی) | Δ | DM p | معنادار؟ | پوشش | پوشش پس از ACI |
|---|---|---|---|---|---|---|---|
| `gp_quantile` | 0.01578 | 0.01335 

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F06_gp_L1.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [14]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F06_gp_L1", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند


📦 بسته‌بندی شد: 506 فایل (52.8 MB خام) → 1 تکه در gpu_outputs/
   gpu_outputs_F06_gp_L1.zip  8.8 MB

بازیابی محلی:
   unzip gpu_outputs_F06_gp_L1.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```